In [0]:
# Quick cleanup: Delete test song 3251 from all layers using Spark SQL
# Executes DELETE statements across Bronze, Silver, and Gold layers
from pyspark.sql.functions import col

test_song_id = 3251

print("="*70)
print("QUICK CLEANUP: DELETING TEST SONG FROM ALL LAYERS")
print("="*70)
print(f"\nTest song: {test_song_id}")
print(f"Purpose: Simulate song never existed in Unity Catalog\n")

# Define all DELETE statements
# Format: (table_name_for_sql, id_column)
delete_statements = [
    # Bronze Layer
    ("acubed.ffr.bronze__songlist", "id"),
    ("acubed.ffr.bronze__playlist", "level"),
    ("acubed.ffr.bronze__charts", "song_id"),
    
    # Silver Layer
    ("acubed.ffr.`silver__chart-features`", "song_id"),
    ("acubed.ffr.`silver__horizontal-density`", "song_id"),
    ("acubed.ffr.silver__notes", "song_id"),
    ("acubed.ffr.`silver__notes-adjusted`", "song_id"),
    ("acubed.ffr.silver__playlist", "song_id"),
    ("acubed.ffr.`silver__vertical-density`", "song_id"),
    
    # Gold Layer
    ("acubed.ffr.gold__features", "song_id"),
]

tables_cleaned = 0
tables_skipped = 0
total_rows_deleted = 0

current_layer = None
for table_name, id_column in delete_statements:
    # Determine layer for section headers
    if "bronze" in table_name and current_layer != "bronze":
        print("\n[BRONZE LAYER]")
        print("-" * 70)
        current_layer = "bronze"
    elif "silver" in table_name and current_layer != "silver":
        print("\n[SILVER LAYER]")
        print("-" * 70)
        current_layer = "silver"
    elif "gold" in table_name and current_layer != "gold":
        print("\n[GOLD LAYER]")
        print("-" * 70)
        current_layer = "gold"
    
    short_name = table_name.replace('acubed.ffr.', '').replace('`', '')
    
    try:
        # Try to check if song exists in the table
        if id_column == "level":
            existing_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE {id_column} = {test_song_id}").first().cnt
        elif id_column == "id":
            existing_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE {id_column} = {test_song_id}").first().cnt
        else:  # song_id
            existing_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE {id_column} = {test_song_id}").first().cnt
        
        if existing_count > 0:
            # Execute DELETE
            spark.sql(f"DELETE FROM {table_name} WHERE {id_column} = {test_song_id}")
            print(f"✓ {short_name:40} - deleted {existing_count} rows")
            tables_cleaned += 1
            total_rows_deleted += existing_count
        else:
            print(f"○ {short_name:40} - song not present (skip)")
            tables_skipped += 1
    except Exception as e:
        # Table doesn't exist or other error
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "cannot be found" in str(e).lower():
            print(f"⊘ {short_name:40} - table doesn't exist (skip)")
            tables_skipped += 1
        else:
            print(f"⚠ {short_name:40} - error: {str(e)[:50]}")
            tables_skipped += 1

print("\n" + "="*70)
print("DELETION SUMMARY")
print("="*70)
print(f"\nTables cleaned: {tables_cleaned}")
print(f"Tables skipped: {tables_skipped}")
print(f"Total rows deleted: {total_rows_deleted}")

if tables_cleaned > 0:
    print(f"\n✅ Cleanup complete - {total_rows_deleted} rows deleted across {tables_cleaned} tables")
else:
    print(f"\n✅ Cleanup complete - song already absent from all tables")

print(f"\n📋 Next Step: Run bronze ingestion to detect song {test_song_id} as NEW")
print(f"   Notebook: ffr__bronze__01a__incremental__ingest-from-api")

In [0]:
# Verify song 3251 status across ALL tables impacted by Cell 2 cleanup
# Run this AFTER bronze ingestion (and optionally silver/gold notebooks)
from pyspark.sql.functions import col

test_song_id = 3251

print("="*70)
print("FULL PIPELINE VERIFICATION")
print("="*70)
print(f"\nVerifying song {test_song_id} across all impacted tables...\n")

# Define all tables that Cell 2 operates on
# Format: (table_name, id_column, layer)
tables_to_verify = [
    # Bronze Layer (3 tables)
    ("acubed.ffr.bronze__songlist", "id", "bronze"),
    ("acubed.ffr.bronze__playlist", "level", "bronze"),
    ("acubed.ffr.bronze__charts", "song_id", "bronze"),
    
    # Silver Layer (6 tables)
    ("acubed.ffr.`silver__chart-features`", "song_id", "silver"),
    ("acubed.ffr.`silver__horizontal-density`", "song_id", "silver"),
    ("acubed.ffr.silver__notes", "song_id", "silver"),
    ("acubed.ffr.`silver__notes-adjusted`", "song_id", "silver"),
    ("acubed.ffr.silver__playlist", "song_id", "silver"),
    ("acubed.ffr.`silver__vertical-density`", "song_id", "silver"),
    
    # Gold Layer (1 table)
    ("acubed.ffr.gold__features", "song_id", "gold"),
]

bronze_success = 0
bronze_missing = 0
silver_success = 0
silver_missing = 0
gold_success = 0
gold_missing = 0

current_layer = None
for table_name, id_column, layer in tables_to_verify:
    # Print layer header
    if layer != current_layer:
        print(f"\n[{layer.upper()} LAYER]")
        print("-" * 70)
        current_layer = layer
    
    short_name = table_name.replace('acubed.ffr.', '').replace('`', '')
    
    try:
        # Check if song exists in the table
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE {id_column} = {test_song_id}").first().cnt
        
        if row_count > 0:
            print(f"✅ {short_name:40} - song exists ({row_count} rows)")
            if layer == "bronze":
                bronze_success += 1
            elif layer == "silver":
                silver_success += 1
            else:
                gold_success += 1
        else:
            print(f"○ {short_name:40} - song not present")
            if layer == "bronze":
                bronze_missing += 1
            elif layer == "silver":
                silver_missing += 1
            else:
                gold_missing += 1
    except Exception as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "cannot be found" in str(e).lower():
            print(f"⊘ {short_name:40} - table doesn't exist")
        else:
            print(f"⚠ {short_name:40} - error: {str(e)[:40]}")
        if layer == "bronze":
            bronze_missing += 1
        elif layer == "silver":
            silver_missing += 1
        else:
            gold_missing += 1

print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print(f"\nBronze: {bronze_success}/3 tables have song {test_song_id}")
print(f"Silver: {silver_success}/6 tables have song {test_song_id}")
print(f"Gold:   {gold_success}/1 tables have song {test_song_id}")
print(f"\nTotal:  {bronze_success + silver_success + gold_success}/10 tables")

# Determine test status
if bronze_success == 3 and silver_success == 0 and gold_success == 0:
    print(f"\n✅ BRONZE INGESTION TEST PASSED")
    print(f"\nBronze layer successfully ingested song {test_song_id}")
    print(f"\n📋 Next Steps:")
    print(f"  • Run silver layer notebooks to propagate to silver tables")
    print(f"  • Run gold layer notebook to generate features")
    print(f"  • Re-run this cell to verify full pipeline")
elif bronze_success == 3 and silver_success == 6 and gold_success == 0:
    print(f"\n✅ SILVER LAYER TEST PASSED")
    print(f"\nSong {test_song_id} successfully propagated through Bronze and Silver")
    print(f"\n📋 Next Step:")
    print(f"  • Run gold layer notebook to generate features")
    print(f"  • Re-run this cell to verify complete")
elif bronze_success == 3 and silver_success == 6 and gold_success == 1:
    print(f"\n✅ FULL PIPELINE TEST PASSED")
    print(f"\nSong {test_song_id} successfully propagated through entire pipeline:")
    print(f"  • Bronze: Ingested from API")
    print(f"  • Silver: Features computed")
    print(f"  • Gold: Final feature vector created")
elif bronze_missing == 3 and silver_missing == 6 and gold_missing == 1:
    print(f"\n✅ CLEANUP VERIFIED")
    print(f"\nSong {test_song_id} successfully removed from all layers")
    print(f"\n📋 Next Step:")
    print(f"  • Run bronze ingestion notebook")
    print(f"  • Expected: 'New songs: 1' (including song {test_song_id})")
    print(f"  • Notebook: ffr__bronze__01a__incremental__ingest-from-api")
else:
    print(f"\n⚠️ PARTIAL STATE DETECTED")
    print(f"\nThe song exists in some layers but not others.")
    print(f"\nDiagnostics:")
    if bronze_missing > 0:
        print(f"  • Bronze: Missing from {bronze_missing} tables - run bronze ingestion")
    if bronze_success > 0 and silver_missing > 0:
        print(f"  • Silver: Missing from {silver_missing} tables - run silver notebooks")
    if silver_success > 0 and gold_missing > 0:
        print(f"  • Gold: Missing from {gold_missing} tables - run gold notebook")
    print(f"\nOr run Cell 2 to clean all layers and start fresh")